# Verify O*NET Tag Accuracy with Job Descriptions

LLM-based O*NET tagging results (`top10_onet_tagged/`) only have titles.
This notebook joins descriptions from the raw CSV so we can verify mapping accuracy.

In [ ]:
import pandas as pd
from pathlib import Path

RAW_CSV = '../../../data/raw/linkedin_postings_2023-2024.csv'
TAGGED_DIR = Path('../../../data/processed/llm_title_accuracy_test/step-2-llm-onet-mapping/top10_onet_tagged')
OUTPUT_DIR = Path('../../../data/processed/llm_title_accuracy_test/step-3-add-descriptions/top10_onet_tagged_with_desc')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

In [2]:
# Load raw CSV - only needed columns
df_raw = pd.read_csv(
    RAW_CSV,
    usecols=['company_name', 'title', 'description'],
    dtype=str
)

print(f'Raw data: {len(df_raw):,} rows')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

Raw data: 123,849 rows
Columns: ['company_name', 'title', 'description']


,company_name,title,description
0,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...
1,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ..."
2,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...
3,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...
4,NaN,Service Technician,Looking for HVAC service tech with experience ...


In [3]:
# Load all tagged CSVs
tagged_files = sorted(TAGGED_DIR.glob('*_tagged.csv'))
tagged_files = [f for f in tagged_files if f.name != 'all_companies_tagged.csv']

print(f'Tagged files: {len(tagged_files)}')
for f in tagged_files:
    print(f'  - {f.name}')

Tagged files: 11
  - Amazon_tagged.csv
  - Apex Systems_tagged.csv
  - capital_one_tagged.csv
  - Dice_tagged.csv
  - Insight Global_tagged.csv
  - Liberty Healthcare and Rehabilitation Services_tagged.csv
  - Macy's_tagged.csv
  - Maxim Healthcare Staffing_tagged.csv
  - TEKsystems_tagged.csv
  - The Job Network_tagged.csv
  - VolunteerMatch_tagged.csv


In [4]:
# Expand semicolon-delimited raw_titles -> individual rows, join with descriptions
all_verified = []

for tagged_file in tagged_files:
    df_tagged = pd.read_csv(tagged_file)
    company = df_tagged['company_name'].iloc[0]
    
    # Expand: one row per (onet_tag, raw_title)
    rows = []
    for _, row in df_tagged.iterrows():
        titles = [t.strip() for t in row['raw_titles'].split(';')]
        for t in titles:
            rows.append({'onet_tag': row['onet_tag'], 'company_name': company, 'raw_title': t})
    df_expanded = pd.DataFrame(rows)
    
    # Get descriptions from raw data for this company
    df_company_raw = df_raw[df_raw['company_name'] == company][['title', 'description']].copy()
    df_company_raw = df_company_raw.drop_duplicates(subset='title', keep='first')
    
    # Join
    df_merged = df_expanded.merge(
        df_company_raw,
        left_on='raw_title',
        right_on='title',
        how='left'
    ).drop(columns=['title'])
    
    matched = df_merged['description'].notna().sum()
    total = len(df_merged)
    print(f'{company}: {matched}/{total} titles matched with descriptions ({matched/total*100:.1f}%)')
    
    # Save per-company
    output_file = OUTPUT_DIR / f'{company.replace("/", "_")}_verified.csv'
    df_merged.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    all_verified.append(df_merged)

print(f'\nSaved to {OUTPUT_DIR}/')

Amazon: 285/286 titles matched with descriptions (99.7%)
Apex Systems: 281/287 titles matched with descriptions (97.9%)
Capital One: 181/183 titles matched with descriptions (98.9%)
Dice: 385/385 titles matched with descriptions (100.0%)
Insight Global: 328/359 titles matched with descriptions (91.4%)
Liberty Healthcare and Rehabilitation Services: 265/265 titles matched with descriptions (100.0%)
Macy's: 306/306 titles matched with descriptions (100.0%)
Maxim Healthcare Staffing: 245/247 titles matched with descriptions (99.2%)
TEKsystems: 395/395 titles matched with descriptions (100.0%)
The Job Network: 706/726 titles matched with descriptions (97.2%)
VolunteerMatch: 292/292 titles matched with descriptions (100.0%)

Saved to top10_onet_tagged_with_desc/


In [5]:
# Combined stats
df_all = pd.concat(all_verified, ignore_index=True)

matched = df_all['description'].notna().sum()
total = len(df_all)

print(f'Total titles: {total:,}')
print(f'With description: {matched:,} ({matched/total*100:.1f}%)')
print(f'Without description: {total - matched:,}')
print(f'\nUnique O*NET tags: {df_all["onet_tag"].nunique()}')
print(f'Companies: {df_all["company_name"].nunique()}')

Total titles: 3,731
With description: 3,669 (98.3%)
Without description: 62

Unique O*NET tags: 375
Companies: 11


In [6]:
# Preview: sample rows with description for verification
sample = df_all[df_all['description'].notna()].groupby('company_name').head(2)
sample['desc_preview'] = sample['description'].str[:150] + '...'

pd.set_option('display.max_colwidth', 80)
sample[['onet_tag', 'company_name', 'raw_title', 'desc_preview']]

,onet_tag,company_name,raw_title,desc_preview
0,Financial Managers,Amazon,"AMZL UTRx Finance Manager, Trans Support","Description\n\nAt Amazon, we're working to be the most customer-centric comp..."
1,Financial Managers,Amazon,"Finance Manager - Finance Operations Accounting, Finance Operations - Accoun...",Description\n\nAmazon is a US-based multinational electronic commerce compan...
286,Natural Sciences Managers,Apex Systems,"(Senior) Director, Drug Discovery (Fungal Biology)","Job#: 2018175\n\nJob Description:\n\n(Senior) Director, Drug Discovery (Fung..."
287,Software Developers,Apex Systems,.NET Developer,.NET Developer 4-5 days on-site in OMAHA12 month contract50-70 an hour \nSr....
573,"Fraud Examiners, Investigators and Analysts",Capital One,"Anti-Money Laundering (AML) Sr. Investigator III, Transaction Monitoring Ope...","West Creek 3 (12073), United States of America, Richmond, VirginiaAnti-Money..."
574,"Fraud Examiners, Investigators and Analysts",Capital One,Anti-Money Laundering Subject Matter Expert (SME) - Special Investigations U...,"West Creek 3 (12073), United States of America, Richmond, VirginiaAnti-Money..."
756,Software Developers,Dice,(only W2) .NET Back End or Full Stack Developer,Dice is the leading career destination for tech experts at every stage of th...
757,Software Developers,Dice,.Net Developer,Dice is the leading career destination for tech experts at every stage of th...
1142,Electrical Engineers,Insight Global,ASIC Synthesis Engineer,Must-haves Following is the Job Description: 10+ years of experience Knowled...
1143,Electrical Engineers,Insight Global,Electrical Design Engineer,"JOB DESCRIPTIONAt Ford Motor Company, we believe freedom of movement drives ..."
